In [ ]:
# Importing Libraries
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist, cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# a. Load and Prepare FashionMNIST Dataset
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Reshape for CNN input — (samples, height, width, channels)
x_train = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_test  = x_test.reshape(x_test.shape[0], 28, 28, 1)

# Normalize pixel values to 0-1
x_train = x_train / 255.0
x_test  = x_test / 255.0

# No to_categorical needed — using sparse_categorical_crossentropy
# labels stay as integers (0–9)

img_rows, img_cols = 28, 28
x_train = x_train.reshape(x_train.shape[0], img_rows, img_cols, 1)
x_test = x_test.reshape(x_test.shape[0], img_rows, img_cols, 1)

# Build CNN Model
model = Sequential()

# First Convolution + Pooling
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Second Convolution + Pooling
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Flatten + Dense layer
model.add(Flatten())
model.add(Dense(128, activation='relu'))

# Output layer — 10 classes, softmax for probabilities
model.add(Dense(10, activation='softmax'))

# Compile using sparse categorical crossentropy
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    x_train, y_train,
    batch_size=128,
    epochs=10,
    verbose=1,
    validation_data=(x_test, y_test)
)

# b. Print model summary
model.summary()

# c. Print accuracy
test_loss, test_acc = model.evaluate(x_test, y_test)
print('Test Accuracy:', test_acc)

# d. Plot Train-Validation Accuracy graph
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Train-Validation Accuracy')

# e. Plot Loss graph
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Train-Validation Loss')

plt.savefig('fashionmnist_results.png')
plt.show()

## Simple Parameter Setting Guide

---

### Conv2D — 4 things to set

---

**1. Filters (the first number)**

```
First Conv layer  → 32
Second Conv layer → 64   (always double)
Third Conv layer  → 128  (keep doubling)
```

Simple dataset (MNIST, FashionMNIST) → start at 32
Complex dataset (CIFAR-10, real photos) → start at 64

---

**2. kernel_size**

```
Image size ≤ 32×32  → always (3,3)
Image size > 128×128 → can try (5,5) or (7,7)
```

Just always use (3,3) — it works for all your datasets.

---

**3. activation**

```
Conv layers and Dense hidden layers → always 'relu'
Output layer → always 'softmax' (for 10 classes)
```

Never changes for your use cases.

---

**4. input_shape**

```
Read directly from x_train.shape

MNIST / FashionMNIST → (28, 28, 1)
CIFAR-10             → (32, 32, 3)
```

Only write this on the **first layer.** Never on any other layer.

---

### MaxPooling2D — 1 thing to set

**pool_size**

```
Always (2,2) — no reason to change this for your datasets
```

It halves the image size each time. That's all it does.

---

### Dense — 2 things to set

**Neurons (the number)**

```
Hidden Dense layer:
  MNIST / FashionMNIST → 128
  CIFAR-10             → 256

Output Dense layer:
  Always = number of classes → 10 for all your datasets
```

---

**activation**

```
Hidden Dense layer → 'relu'
Output Dense layer → 'softmax'
```

---

### The 3 questions to ask yourself:

```
1. What dataset?
   → gives you input_shape and num_classes

2. Which layer number is this Conv?
   → gives you the filter count (32 → 64 → 128)

3. Is this a hidden layer or output layer?
   → hidden = relu
   → output = softmax
```

---

### Filled in for each dataset:

**MNIST / FashionMNIST:**
```python
Conv2D(32,  (3,3), activation='relu', input_shape=(28,28,1))
MaxPooling2D((2,2))
Conv2D(64,  (3,3), activation='relu')
MaxPooling2D((2,2))
Flatten()
Dense(128, activation='relu')
Dense(10,  activation='softmax')
```

**CIFAR-10:**
```python
Conv2D(64,  (3,3), activation='relu', input_shape=(32,32,3))
MaxPooling2D((2,2))
Conv2D(128, (3,3), activation='relu')
MaxPooling2D((2,2))
Flatten()
Dense(256, activation='relu')
Dense(10,  activation='softmax')
```

The **only things that changed** between the two:
- `input_shape` → from dataset
- Starting filters → 32 for simple, 64 for complex
- Dense neurons → 128 for simple, 256 for complex

Everything else stays exactly the same.

## Complete Parameter Selection Guide

---

## What You Need to Identify Before Writing Any Code

---

### STEP 1 — Look at Your Data

Before touching the model, answer these questions about your dataset:

---

**Question 1 — What is the image size?**

Print `x_train.shape` and read the numbers.

```
(60000, 28, 28)     → images are 28×28
(50000, 32, 32, 3)  → images are 32×32
(1000, 224, 224, 3) → images are 224×224
```

This tells you:
- What to put in `input_shape`
- How many Conv+Pool layers you can add before the image becomes too small
- What kernel size is appropriate

---

**Question 2 — How many channels?**

Look at the last number in the shape.

```
(60000, 28, 28)    → no channel shown → grayscale → use 1
(60000, 28, 28, 1) → explicitly 1 → grayscale → use 1
(50000, 32, 32, 3) → 3 channels → color RGB → use 3
```

This goes into `input_shape` as the third value: `(height, width, channels)`

---

**Question 3 — How many classes?**

Print `len(np.unique(y_train))` and count the unique labels.

```
2 classes  → binary problem  → output layer needs 1 neuron + sigmoid
10 classes → multi-class     → output layer needs 10 neurons + softmax
100 classes → multi-class    → output layer needs 100 neurons + softmax
```

This directly sets:
- Number of neurons in the output Dense layer
- Which activation to use in output layer
- Which loss function to use

---

**Question 4 — How complex are the images?**

Look at the images visually.

```
Simple images:
→ plain background, clear object, no texture detail
→ examples: MNIST digits, simple shapes
→ needs fewer layers and filters

Complex images:
→ real world photos, multiple objects, textures, backgrounds
→ examples: CIFAR-10, animals, vehicles
→ needs more layers and more filters
```

Complexity decides:
- How many Conv layers to stack
- How many filters per layer
- How many Dense neurons

---

**Question 5 — How large is the dataset?**

Count `x_train.shape[0]`

```
Small dataset (< 5,000 images)
→ use fewer layers — complex model will overfit
→ use Dropout heavily
→ consider data augmentation

Medium dataset (5,000 – 50,000 images)
→ standard CNN works fine

Large dataset (> 50,000 images)
→ can afford deeper, wider model
→ needs larger batch size to train efficiently
```

---

---

## STEP 2 — Deciding Each Parameter

---

### `input_shape`

**What to identify:** Image height, width, and number of channels from `x_train.shape`

```
Rule:
→ Take height and width directly from the shape
→ If no channel dimension shown, add 1 (grayscale)
→ If 3 shown, it's color

Small image (28×28 or 32×32)  → simple images
Large image (224×224+)         → complex images, more detail
```

---

### Number of Conv Layers

**What to identify:** Image size and complexity

```
Guiding rule — each MaxPooling halves the image size.
You cannot keep pooling once the image becomes too small (below 4×4 is pointless)

Image 28×28:
→ After 1st pool: 14×14
→ After 2nd pool: 7×7
→ After 3rd pool: 3×3  ← getting too small
→ Maximum: 2 pooling layers is safe

Image 32×32:
→ After 1st pool: 16×16
→ After 2nd pool: 8×8
→ After 3rd pool: 4×4  ← borderline
→ Maximum: 2 to 3 pooling layers

Image 224×224:
→ Can afford 5 pooling layers comfortably (like VGG)

Simple dataset → 2 Conv+Pool blocks is enough
Complex dataset → 3 to 5 Conv+Pool blocks
```

---

### Number of Filters in Conv2D

**What to identify:** Image complexity and which layer you are adding

```
Two things decide this:

1. Complexity of dataset:
   Simple (MNIST)        → start at 32
   Medium (FashionMNIST) → start at 32 or 64
   Complex (CIFAR-10)    → start at 64

2. Position of the layer:
   Always double filters as you go deeper
   First layer  → starting value (32 or 64)
   Second layer → double it (64 or 128)
   Third layer  → double again (128 or 256)

Why double?
Early layers detect simple things (edges, lines) — fewer filters needed
Deeper layers detect complex things (shapes, objects) — more filters needed
```

---

### Kernel Size in Conv2D

**What to identify:** Image size

```
Small images (28×28 to 32×32):
→ always use (3,3)
→ larger kernels cover too much of the small image at once

Medium images (64×64 to 128×128):
→ (3,3) is still safe default
→ (5,5) can be tried

Large images (224×224 and above):
→ (3,3) for regular CNN and VGG
→ (7,7) or (11,11) for AlexNet-style first layer only

Key point:
The kernel must always be smaller than the image.
A 5×5 kernel on a 4×4 image makes no sense.
```

---

### Padding

**What to identify:** Whether you want the image size preserved after Conv

```
'valid' (no padding):
→ image shrinks after every Conv layer
→ fine if you have large images and plan to pool anyway

'same' (adds zeros around border):
→ image stays same size after Conv
→ use this for small images (28×28, 32×32) so they don't shrink too fast
→ safer default for all cases
```

---

### Pool Size in MaxPooling2D

**What to identify:** How aggressively you want to reduce size

```
(2,2) → halves the image → standard default for all cases
(3,3) → reduces to one-third → only for large images
(1,1) → no reduction → pointless unless you want overlapping pooling (AlexNet style)

For MNIST, FashionMNIST, CIFAR-10:
→ always (2,2), no reason to change this
```

---

### Dense Layer Neurons

**What to identify:** Image complexity and dataset size

```
The Dense layer after Flatten needs enough neurons to learn
the patterns extracted by Conv layers.

Too few neurons → can't learn complex patterns → low accuracy
Too many neurons → overfits → learns training data but fails on test data

Simple dataset, small images:
→ 64 to 128 neurons is enough

Medium complexity:
→ 128 to 256 neurons

Complex dataset, large images:
→ 256 to 512 neurons

Always use powers of 2:
64, 128, 256, 512, 1024
These are convention and work well with hardware optimization.
```

---

### Dropout Rate

**What to identify:** Whether your model is overfitting

```
Dropout randomly turns off neurons during training to prevent overfitting.

When to use more Dropout:
→ training accuracy much higher than validation accuracy (overfitting)
→ small dataset
→ large Dense layers

When to use less Dropout:
→ training and validation accuracy are close
→ large dataset
→ model is already underfitting (low training accuracy)

Rate guide:
0.25 → light regularization — small models, large datasets
0.4  → medium — general purpose
0.5  → strong — large Dense layers, small datasets, clear overfitting
```

---

### Loss Function

**What to identify:** Number of classes and label format

```
2 classes (binary):
→ loss = 'binary_crossentropy'
→ output layer = Dense(1, activation='sigmoid')

More than 2 classes + labels are one-hot encoded (used to_categorical):
→ loss = 'categorical_crossentropy'
→ output layer = Dense(num_classes, activation='softmax')

More than 2 classes + labels are plain integers (NOT one-hot):
→ loss = 'sparse_categorical_crossentropy'
→ output layer = Dense(num_classes, activation='softmax')
→ no to_categorical needed
```

---

### Optimizer and Learning Rate

**What to identify:** Model size and training behavior

```
Which optimizer:
→ Always start with Adam — it works well in almost all cases
→ SGD is an alternative but needs more manual tuning

Learning rate for Adam:
→ Default 0.001 works for most cases — start here
→ If loss barely decreases → try higher (0.01)
→ If loss jumps around unstably → try lower (0.0001)
→ Large models like VGG and AlexNet → use lower (0.0001)
```

---

### Batch Size

**What to identify:** Dataset size and available memory

```
What batch size affects:
→ How many images are processed before weights are updated
→ Memory usage
→ Training speed

Small dataset (< 10,000):
→ 32

Medium dataset (10,000 – 60,000):
→ 64 to 128

Large dataset (> 60,000):
→ 128 to 256

If you get memory error:
→ halve the batch size

If training is very slow:
→ double the batch size
```

---

### Epochs

**What to identify:** When accuracy stops improving

```
Epochs = how many times the model sees the entire dataset

Too few:
→ underfitting — model hasn't learned enough

Too many:
→ overfitting — model memorizes training data

Starting points:
Simple dataset (MNIST)       → 10 epochs
Medium dataset (FashionMNIST) → 20 epochs
Complex dataset (CIFAR-10)   → 50 epochs

The real rule:
Watch validation accuracy during training.
When validation accuracy stops improving for several epochs → stop.
This is called Early Stopping.
```

---

---

## STEP 3 — The Questions Checklist

Every time you face a new dataset, answer these questions in order:

---

```
About the data:
□ What is x_train.shape?           → sets input_shape
□ How many channels?               → sets 1 or 3 in input_shape
□ How many unique classes?         → sets output neurons and loss function
□ Are labels integers or one-hot?  → sets which loss function to use
□ What is pixel range (0-255)?     → sets whether to normalize
□ How many training samples?       → sets batch size and dropout

About the images:
□ Simple or complex patterns?      → sets number of layers and filters
□ What is the image size?          → sets kernel size and max pooling layers allowed

About the model behavior (after first training):
□ Is training accuracy low?        → model too simple, add layers/filters
□ Is val accuracy much lower       
  than training accuracy?          → overfitting, add dropout or reduce model size
□ Is loss not decreasing?          → wrong learning rate or preprocessing issue
```

---

## Summary Table — What Identifies What

| Parameter | Identified by |
|-----------|--------------|
| `input_shape` | `x_train.shape` |
| Output layer neurons | Number of unique classes |
| Output activation | Binary → sigmoid, Multi-class → softmax |
| Loss function | Number of classes + label format |
| Number of Conv layers | Image size — how many times you can halve |
| Filters per layer | Complexity + double each layer |
| Kernel size | Image size — (3,3) for small, larger for big images |
| Padding | Image size — 'same' for small images |
| Pool size | Almost always (2,2) |
| Dense neurons | Complexity — 128 for simple, 512 for complex |
| Dropout rate | Overfitting — more overfit = higher dropout |
| Batch size | Dataset size and memory |
| Epochs | When validation accuracy stops improving |
| Learning rate | Start 0.001 — adjust if unstable or too slow |